# Tutorial: Validating causal abstractions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MelouxM/CAE/blob/main/tutorial.ipynb)

This notebook is a hands-on introduction to **`causal_abstraction`**, the library that accompanies
the paper *"Validating Causal Abstraction Metrics on Simulated Complex Systems"*.

The central question the library answers is: *when is a simple high-level causal model a **valid**
explanation of a complex low-level system?* We formalize this as **constructive causal abstraction**.
A low-level structural causal model `M` is abstracted by a high-level model `E` through a map `τ`,
which has two parts:

* a **coarse-graining** `a` that groups low-level (micro) variables into high-level (macro) variables,
  and
* a **value map** `τ_X` that translates micro-states into macro-labels and back.

Validity is checked on a **commuting diagram**: intervening on `E` and reading the result should match
grounding that same intervention into `M`, running `M`, and abstracting the result back up. The paper's
metric for this is the **Causal Abstraction Error (CAE)**, which comes in two directions:

* **`CAE↑`** (bottom-up): micro-interventions are sampled in `M`'s space and lifted through `τ`, and
* **`CAE↓`** (top-down): macro-interventions are sampled in `E` and grounded into `M`.

Each runs in a **faithful** form (the canonical CAE, which also perturbs the *unmapped* variables `Φ`
to detect leakage) and a **non-faithful** ablation. A score of **0 means a perfect abstraction**;
larger values mean more error.

The notebook has three parts:

1. **Basic evaluation**: build a tiny abstraction from scratch and score it.
2. **Benchmark systems**: load one of the ready-made systems shipped with the library.
3. **Baseline metrics**: run some of the other candidate metrics the paper compares CAE against.

The paper is the source of truth for all notation; this notebook mirrors it.

> **Running in Google Colab?** Just run the cells in order. The first code cell
> (*Setup*) clones the repository and installs the library for you; the benchmark
> systems used in Parts 2–3 live in the repo (not the pip package), so the clone is
> required. When you run this notebook locally inside a checkout of the repo, that
> cell detects the checkout and does nothing.

In [ ]:
# === Setup: run this first ===================================================
# If you opened this notebook in Google Colab straight from GitHub, the runtime
# has only the .ipynb file: the `causal_abstraction` library is not installed and
# the `systems/` and `test/` directories that Parts 2-3 load by path are absent
# (they are deliberately not shipped in the pip package, only in the repository).
# This cell detects that case, clones the repository, and installs the library.
# When a checkout is already present (e.g. running locally inside the repo) it
# does nothing.
import os
import subprocess
import sys


def _has_checkout(start="."):
    """True if `start` or any ancestor directory contains the causal_abstraction package."""
    path = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(path, "causal_abstraction")):
            return True
        parent = os.path.dirname(path)
        if parent == path:
            return False
        path = parent


if not _has_checkout():
    repo_url = "https://github.com/MelouxM/CAE.git"
    clone_dir = os.path.abspath("CAE")
    if not os.path.isdir(os.path.join(clone_dir, "causal_abstraction")):
        subprocess.run(["git", "clone", "--depth", "1", repo_url, clone_dir], check=True)
    os.chdir(clone_dir)  # so the next cell locates the repo root, systems/ and test/
    # Core dependencies only; the tutorial needs none of the optional extras.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "."], check=True)
    print("Set up causal_abstraction in", clone_dir)

In [1]:
import os
os.environ.setdefault("TQDM_DISABLE", "1")  # keep the saved notebook output clean
import sys
import logging

import numpy as np

logging.getLogger("causal_abstraction").setLevel(logging.WARNING)  # silence per-task INFO logs

# Make the repository importable when running from anywhere in the repo.
_REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(_REPO_ROOT, "causal_abstraction")):
    parent = os.path.dirname(_REPO_ROOT)
    if parent == _REPO_ROOT:
        raise RuntimeError("Could not locate the repository root.")
    _REPO_ROOT = parent
for _p in (_REPO_ROOT, os.path.join(_REPO_ROOT, "systems")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import causal_abstraction as ca

print("causal_abstraction loaded; public API has", len(ca.__all__), "names")

causal_abstraction loaded; public API has 75 names


## Part 1: A basic evaluation from scratch

To evaluate an abstraction you assemble five ingredients. We build the smallest possible example: a
low-level "AND gate" that operates on noisy voltages, abstracted by a high-level boolean `Y = A AND B`.

| Ingredient | Symbol | Class |
|---|---|---|
| Micro-variable layout | (none) | `MicroVariableSchema` |
| Coarse-graining (which micro-vars form each macro-var; defines `Φ`) | `a` | `CoarseGrainingMap` |
| Value map (micro-state ↔ macro-label) | `τ_X` | `ValueMap` |
| Low-level model (the complex system) | `M` | `LowLevelModel` |
| High-level model (the explanation) | `E` | `CausalGraph` |

Our low-level system has four micro-variables: the inputs `a`, `b`, the output `out`, and a `scratch`
register. `scratch` is deliberately **causally inert**: the output never depends on it. Because we
will not map it to any high-level variable, the library treats it as an **unmapped variable** `Φ`,
and the faithful CAE will perturb it to confirm it really has no effect.

In [2]:
from causal_abstraction import (
    SystemState, LowLevelModel, CausalGraph,
    MicroVariableSchema, CoarseGrainingMap, ValueMap, RectSubspace,
)


class AndGate(LowLevelModel):
    # Low-level model M: out = (a AND b) on noisy voltages; `scratch` is inert (a Phi variable).

    MICRO_VARS = ["a", "b", "out", "scratch"]

    def forward_with_interventions(self, input_state: SystemState, interventions: dict) -> SystemState:
        values = dict(input_state.values)
        values.update(interventions)

        # Infer the batch size from any 2-D intervention array.
        batch = 1
        for v in values.values():
            arr = np.asarray(v, dtype=float)
            if arr.ndim >= 2:
                batch = max(batch, arr.shape[0])

        def column(name, default=0.0):
            if name not in values:
                return np.full((batch, 1), default)
            arr = np.asarray(values[name], dtype=float).reshape(-1, 1)
            if arr.shape[0] == 1 and batch > 1:
                arr = np.broadcast_to(arr, (batch, 1)).copy()
            return arr

        a = column("a")
        b = column("b")
        # Honour an intervention that forces `out`; otherwise compute it from a, b only.
        if "out" in interventions:
            out = column("out")
        else:
            out = ((a > 0.5) & (b > 0.5)).astype(float)
        scratch = column("scratch")  # carried through but never used -> faithful

        return SystemState(values={"a": a, "b": b, "out": out, "scratch": scratch})


schema = MicroVariableSchema.from_names(AndGate.MICRO_VARS)

# Coarse-graining a: each macro-variable is backed by one micro-variable.
# `scratch` is left out, so it becomes the unmapped set Phi.
cg = CoarseGrainingMap(schema, {"A": ["a"], "B": ["b"], "Y": ["out"]})
print("Unmapped (Phi) variables:", cg.phi_variables)

# Value map tau_X: voltages near 0 -> label 0, voltages near 1 -> label 1.
LOW = RectSubspace((-0.5, 0.5))
HIGH = RectSubspace((0.5, 1.5))
value_specs = {name: {0: LOW, 1: HIGH} for name in ("A", "B", "Y")}
vm = ValueMap(cg, value_specs)
print("Value map surjective:", vm.validate_surjectivity())

Unmapped (Phi) variables: {'scratch'}
Value map surjective: True


Now the high-level model `E`. We write two versions of it: a **correct** one (`Y = A AND B`) and a
**wrong** one (`Y = A OR B`). The wrong model is a deliberately invalid abstraction that the CAE
should flag. Each variable carries a `distribution` so the samplers know how to draw interventions on
it.

In [3]:
BIT = {0: 0.5, 1: 0.5}


def build_high_level(kind: str) -> CausalGraph:
    # kind='and' is the correct abstraction; kind='or' is a wrong one.
    op = (lambda A, B: int(bool(A) and bool(B))) if kind == "and" else (lambda A, B: int(bool(A) or bool(B)))
    E = CausalGraph()
    E.add_variable("A", lambda: None, distribution=BIT)
    E.add_variable("B", lambda: None, distribution=BIT)
    E.add_variable("Y", op, parents=["A", "B"], distribution=BIT)
    return E


# Sanity check: the high-level equation behaves as intended.
E_and = build_high_level("and")
for a in (0, 1):
    for b in (0, 1):
        print(f"A={a} B={b} -> Y={E_and.predict({'A': a, 'B': b})['Y']}")

A=0 B=0 -> Y=0
A=0 B=1 -> Y=0
A=1 B=0 -> Y=0
A=1 B=1 -> Y=1


With all five ingredients we build an `EvaluationEngine` and run the four standard causal-abstraction
tasks via `StandardTasks`:

* `CAE_up` / `CAE_down`: the faithful CAE in each direction (these also perturb `Φ`), and
* `CAE_up_nf` / `CAE_down_nf`: the non-faithful ablations.

We use `metric="hard"` (exact-match dissimilarity) because the labels here are discrete.

In [4]:
from causal_abstraction import (
    EvaluationConfig, EvaluationEngine, TopDownSampler, BottomUpSampler,
)
from causal_abstraction.tasks import StandardTasks

low_level = AndGate()


def evaluate(high_level, *, seed=0):
    cfg = EvaluationConfig(metric="hard", seed=seed)
    engine = EvaluationEngine(high_level, low_level, vm, cg, cfg)
    td = TopDownSampler(vm)
    bu = BottomUpSampler(vm)
    tasks = [
        StandardTasks.score(engine.builder, sampler=td, name="CAE_down", include_faithfulness=True),
        StandardTasks.score(engine.builder, sampler=bu, name="CAE_up", include_faithfulness=True),
        StandardTasks.score(engine.builder, sampler=td, name="CAE_down_nf"),
        StandardTasks.score(engine.builder, sampler=bu, name="CAE_up_nf"),
    ]
    return engine.run_tasks(tasks, n_samples=300, batch_size=30, max_interventions=2)


print("Correct abstraction (Y = A AND B):")
print(evaluate(build_high_level("and")))

Correct abstraction (Y = A AND B):
Results:
  CAE_down            : Score = 0.0000 ± 0.00e+00 / 0.0 ↓
  CAE_up              : Score = 0.0000 ± 0.00e+00 / 0.0 ↓
  CAE_down_nf         : Score = 0.0000 ± 0.00e+00 / 0.0 ↓
  CAE_up_nf           : Score = 0.0000 ± 0.00e+00 / 0.0 ↓


The correct abstraction scores ≈ 0 on every task: the high-level model and the grounded low-level
model agree, and perturbing the `Φ` variable `scratch` does not change the output, so the faithful
scores stay low too.

Now the wrong abstraction:

In [5]:
print("Wrong abstraction (Y = A OR B):")
print(evaluate(build_high_level("or")))

Wrong abstraction (Y = A OR B):


Results:
  CAE_down            : Score = 0.4667 ± 4.62e-02 / 0.0 ↓
  CAE_up              : Score = 0.6200 ± 8.42e-02 / 0.0 ↓
  CAE_down_nf         : Score = 0.4667 ± 7.45e-02 / 0.0 ↓
  CAE_up_nf           : Score = 0.4933 ± 1.12e-01 / 0.0 ↓


The wrong model scores clearly **higher**: `A OR B` disagrees with the gate whenever exactly one input
is high, and the CAE picks that up. That gap between a valid and an invalid abstraction is exactly the
signal the metric is designed to produce.

A `TaskResults` object also exposes the raw numbers programmatically:

In [6]:
valid_scores = evaluate(build_high_level("and")).summary()
wrong_scores = evaluate(build_high_level("or")).summary()
print(f"{'task':<14}{'valid':>10}{'wrong':>10}")
for task in valid_scores:
    print(f"{task:<14}{valid_scores[task]:>10.4f}{wrong_scores[task]:>10.4f}")

task               valid     wrong
CAE_down          0.0000    0.4667
CAE_up            0.0000    0.6200
CAE_down_nf       0.0000    0.4667
CAE_up_nf         0.0000    0.4933


## Part 2: Using a provided benchmark system

You rarely need to build a system by hand: the library ships a benchmark of ten complex systems plus a
suite of controlled experiments, each with a manually constructed ground-truth abstraction `(M, E, τ)`
and **invalid contrastive conditions** that a good metric should detect. Each system lives in
`systems/NN_*.py` and is evaluated by a matching suite in `test/NN_*.py`.

We load the simplest one, the **2-bit adder logic circuit** (`systems/01_logic_circuit.py`), the same
way the suites do, via the shared `load_system` helper in `test/utils.py`. (The benchmark modules use
numeric prefixes like `01_logic_circuit.py`, which are not importable through normal package mechanics,
so they are loaded by file path.) The module provides builder functions for the low-level netlist
simulator, the coarse-graining and value maps, and several high-level models, including a `valid` one
and a `failing` one (which uses OR instead of XOR for the sum bit).

In [7]:
# The evaluation suites share a helper, test/utils.py, whose load_system() imports a
# numeric-prefixed benchmark module (01_logic_circuit.py, ...) by file path. We load
# that helper by its own file path, not via `from utils import load_system`, so
# the notebook is robust even if an unrelated `utils.py` sits earlier on sys.path.
import importlib.util as _ilu

_spec = _ilu.spec_from_file_location("ca_suite_utils", os.path.join(_REPO_ROOT, "test", "utils.py"))
_suite_utils = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_suite_utils)
load_system = _suite_utils.load_system

lc = load_system("01_logic_circuit.py", "logic_circuit")

gates, all_wires = lc.build_2bit_adder()
lc_schema = MicroVariableSchema.from_names(all_wires)
lc_low_level = lc.NetlistSimulator(gates, all_wires)        # the low-level model M
lc_cg, lc_vm = lc.build_cg_and_vm(lc_schema)                # the coarse-graining a and value map tau

high_level_valid = lc.build_valid_high_level_model()        # correct high-level model E
high_level_failing = lc.build_failing_high_level_model()    # an invalid one (OR instead of XOR)

print("Micro-variables (wires):", sorted(all_wires))
print("Macro-variables:", list(high_level_valid.variables))
print("Internal variables:", sorted(lc_cg.internal_variables))
print("Unmapped (Phi) variables:", sorted(lc_cg.phi_variables))

Micro-variables (wires): ['A0', 'A1', 'B0', 'B1', 'C0', 'C1', 'C2', 'S0', 'S1', 'bit0_and1', 'bit0_and2', 'bit0_xor1', 'bit1_and1', 'bit1_and2', 'bit1_xor1']
Macro-variables: ['Operand_A', 'Operand_B', 'Carry_In', 'Internal_Carries', 'Result_Sum', 'Result_Carry']
Internal variables: ['bit0_and1', 'bit0_and2', 'bit0_xor1', 'bit1_and1', 'bit1_and2', 'bit1_xor1']
Unmapped (Phi) variables: []


The adder has a richer structure than Part 1: it has **internal variables** (the intermediate
logic gates), which are part of the mechanism but are not intervened on, distinct from `Φ`. We score
the same four CAE tasks on the valid and the failing high-level models.

In [8]:
OUTPUTS = ["Result_Sum", "Result_Carry"]


def evaluate_logic(high_level, *, seed=0):
    cfg = EvaluationConfig(metric="hard", seed=seed)
    engine = EvaluationEngine(high_level, lc_low_level, lc_vm, lc_cg, cfg)
    td = TopDownSampler(lc_vm)
    bu = BottomUpSampler(lc_vm)
    tasks = [
        StandardTasks.score(engine.builder, sampler=td, name="CAE_down", include_faithfulness=True),
        StandardTasks.score(engine.builder, sampler=bu, name="CAE_up", include_faithfulness=True),
        StandardTasks.score(engine.builder, sampler=td, name="CAE_down_nf"),
        StandardTasks.score(engine.builder, sampler=bu, name="CAE_up_nf"),
    ]
    return engine, engine.run_tasks(tasks, n_samples=300, batch_size=30, max_interventions=2)


engine_valid, res_valid = evaluate_logic(high_level_valid)
engine_failing, res_failing = evaluate_logic(high_level_failing)

valid_scores = res_valid.summary()
failing_scores = res_failing.summary()
print(f"{'task':<14}{'valid':>10}{'failing':>10}")
for task in valid_scores:
    print(f"{task:<14}{valid_scores[task]:>10.4f}{failing_scores[task]:>10.4f}")

task               valid   failing
CAE_down          0.0000    0.2048
CAE_up            0.0000    0.1984
CAE_down_nf       0.0000    0.2093
CAE_up_nf         0.0000    0.2000


As with the toy example, the valid abstraction scores near zero while the failing one scores higher,
on a real benchmark system this time, with no abstraction code written by hand.

## Part 3: Baseline metrics

CAE is one of **30+ candidate metrics** the paper benchmarks, drawn from observational, functional,
information-theoretic, and causal families. The library exposes the baselines two ways:

* **Observational tasks** (`StandardTasks.observational_*`): compare high- and low-level outputs under
  input-only interventions, scored with a dissimilarity such as MSE or R².
* **Analytical metrics** (`AnalyticalMetric` subclasses such as `IIAMetric`, `SymbionMetric`,
  `BCCMetric`): compute a property directly from `M`, `E`, and `τ` via
  `engine.run_analytical_metrics`, which returns an `AnalyticalResults` mapping: a `dict` of
  `{metric_name: score}` that also reports any metric that failed via `.errors` / `.failed`.

As reported by `summary()`, all of these are dissimilarities normalized so that **0 is a perfect match
and larger is worse**. We reuse the logic-circuit engines built in Part 2.

In [9]:
# Observational baselines: only the output variables are scored.
def observational(engine, *, seed=0):
    bu = BottomUpSampler(lc_vm)
    tasks = [
        StandardTasks.observational_mse(engine.builder, sampler=bu, output_vars=OUTPUTS),
        StandardTasks.observational_r2(engine.builder, sampler=bu, output_vars=OUTPUTS),
    ]
    return engine.run_tasks(tasks, n_samples=300, batch_size=30).summary()


obs_valid = observational(engine_valid)
obs_failing = observational(engine_failing)
print(f"{'baseline':<14}{'valid':>10}{'failing':>10}")
for name in obs_valid:
    print(f"{name:<14}{obs_valid[name]:>10.4f}{obs_failing[name]:>10.4f}")

baseline           valid   failing
MSE               0.0000    0.4953
R^2               0.0000    0.5000


In [10]:
from causal_abstraction import IIAMetric, SymbionMetric, BCCMetric

analytical = [
    IIAMetric(inner_metric="mse", output_vars=OUTPUTS, n_pairs=100),
    SymbionMetric(output_vars=OUTPUTS),
    BCCMetric(n_pairs=100, inner_metric="mse"),
]

bu = BottomUpSampler(lc_vm)
an_valid = engine_valid.run_analytical_metrics(analytical, sampler=bu, n_samples=200)
an_failing = engine_failing.run_analytical_metrics(analytical, sampler=bu, n_samples=200)

# an_valid is an AnalyticalResults mapping; .failed lists any metric that raised.
if an_valid.failed or an_failing.failed:
    print("metrics that failed:", sorted(set(an_valid.failed) | set(an_failing.failed)))

print(f"{'metric':<24}{'valid':>10}{'failing':>10}")
for name in an_valid:
    if an_valid.is_error(name) or an_failing.is_error(name):
        print(f"{name:<24}{'(failed)':>10}")
    else:
        print(f"{name:<24}{an_valid[name]:>10.4f}{an_failing[name]:>10.4f}")

metric                       valid   failing
IIAMetric                   0.0000    0.3033
SymbionMetric               0.0000    0.5625
BCCMetric                   0.0000    0.0000


Each metric returns a score for the valid and the failing abstraction. Comparing the two columns is how
the paper measures a metric's **discriminative power**: its ability to separate a valid abstraction
from an invalid one. Here `IIAMetric` and `SymbionMetric` both score the failing model well above the
valid one, but `BCCMetric` gives ≈ 0 for *both*: on this system it cannot tell them apart. That is the
crux of the paper's finding: across the full benchmark, many baselines that look fine on one system
fail on others, and only the causal-abstraction metrics *with* faithfulness testing (the faithful
`CAE↑`/`CAE↓`) discriminate reliably.

To reproduce those aggregate numbers, run the full suites (e.g. `python test/01_logic_circuit.py`),
which repeat these measurements across many conditions and accumulate statistics.

### Where to go next

* `systems/NN_*.py`: the ten benchmark systems and the controlled experiments, each a worked
  `(M, E, τ)` example.
* `test/NN_*.py`: the evaluation suites that run the full metric battery across conditions.
* The paper: the formal definitions of `M`, `E`, `τ`, `a`, `Φ`, and the CAE (the source of truth for
  all notation).